# SI4006 · Sesión 13 · Lab: **Integrar la visión a SU sistema — y medirla**  ·  RESUELTO ✅

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 4 · Entrega M4

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s13/S13_Lab_Integracion_Visual_RESUELTO.ipynb)

---

Hoy no hay modelo nuevo. Hay una receta de cinco pasos, repartida en dos partes:

- **Parte 1 · Integrar:** eligen el patrón (input / tool / output), construyen `ver()` con el modelo que corresponde, la prueban sola y la enchufan: `sistema_multimodal(pregunta, imagen)` responde de punta a punta.
- **Parte 2 · Medir:** escriben ≥5 casos con imagen, completan los criterios del harness, corren todo y llenan la **tabla de delta contra M3**. Exportan el scorecard M4.

> **Traigan:** 3–6 imágenes de SU dominio, su `responder_m3` (o la URL de su repo) y el scorecard de M3.
>
> Corre en Colab con GPU **T4**. El patrón *tool* con CLIP corre en CPU; *input* (VLM) y *output* (difusión) necesitan la T4.

Este notebook terminado es el núcleo de la entrega **M4** (cierra dom 25 OCT · 10 %).

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.45" accelerate pillow qwen-vl-utils pandas
# solo si su patrón es OUTPUT (difusión):
# %pip install -q diffusers
print('Listo.')

In [ ]:
import torch, requests, io, json, re, time
import pandas as pd
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

def cargar(src):
    if str(src).startswith('http'):
        return Image.open(io.BytesIO(requests.get(src, timeout=30).content)).convert('RGB')
    return Image.open(src).convert('RGB')

## 1 · Su patrón y sus imágenes

Una sola decisión antes de escribir código. Si dudan, relean la slide *Los tres patrones · decisión final*: el patrón correcto es el que responde la pregunta de su usuario con menos piezas nuevas.

In [ ]:
# 👉 TODO 1 · elijan UNO: 'input' (el usuario manda una foto, un VLM la lee)
#                       'tool'  (el agente decide mirar: CLIP / ViT clasifica)
#                       'output' (el sistema dibuja: difusión)
PATRON = 'input'   # (resuelto con 'input'; cambien para probar 'tool' u 'output')

# 👉 la pregunta de su usuario que necesita ojos (una frase):
PREGUNTA_USUARIO = '¿Qué le pasa a esta hoja de café?'   # ← cambien

# 👉 SUS IMÁGENES: {'nombre corto': 'URL o ruta'}. Las de ejemplo (COCO) son para que corra solo.
IMAGENES = {
    'gatos': 'http://images.cocodataset.org/val2017/000000039769.jpg',
    'sala':  'http://images.cocodataset.org/val2017/000000000139.jpg',
    'esqui': 'http://images.cocodataset.org/val2017/000000000785.jpg',
}
imgs = {k: cargar(v) for k, v in IMAGENES.items()}
assert PATRON in ('input', 'tool', 'output')
print('patrón:', PATRON, '· imágenes:', list(imgs))

## 2 · Parte 1 · La función `ver()`

Una función, una responsabilidad: **imagen → algo que su sistema de M3 ya sabe consumir** (texto, JSON, una clase) — o, en *output*, **prompt → imagen**. Corran solo la sección de su patrón; las otras pueden quedar sin ejecutar.

### 2A · Patrón INPUT · un VLM lee la imagen (Qwen2-VL-2B, S11)

In [ ]:
if PATRON == 'input':
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
    VLM_ID = 'Qwen/Qwen2-VL-2B-Instruct'
    vlm = Qwen2VLForConditionalGeneration.from_pretrained(VLM_ID, torch_dtype=torch.float16 if device == 'cuda' else torch.float32, device_map='auto')
    vproc = AutoProcessor.from_pretrained(VLM_ID)

    @torch.no_grad()
    def preguntar(imagen, pregunta, system=None, max_new_tokens=160):
        msgs = ([{'role': 'system', 'content': system}] if system else []) + \
               [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': pregunta}]}]
        texto = vproc.apply_chat_template(msgs, add_generation_prompt=True)
        ent = vproc(text=[texto], images=[imagen], return_tensors='pt').to(vlm.device)
        out = vlm.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False)
        return vproc.batch_decode(out[:, ent['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()
    print('VLM listo')

In [ ]:
if PATRON == 'input':
    VALVULA = ('Responde SOLO con lo que se ve en la imagen. '
               'Si algo no se aprecia, responde exactamente: "No se aprecia en la imagen."')

    # 👉 TODO 2 · la instrucción de SU dominio: qué debe extraer el VLM y en qué formato (JSON con campos fijos).
    INSTRUCCION_DOMINIO = ('Describe la imagen en dos frases, en español, y luego devuelve SOLO un JSON con los campos '
                           '{"objetos": [...], "escena": "interior|exterior", "texto_visible": "texto o null"}')   # ← cambien

    def parsear_json(t):
        i, j = t.find('{'), t.rfind('}')
        if i == -1 or j == -1: return None
        try: return json.loads(t[i:j+1])
        except Exception: return None

    def ver(imagen, instruccion=None):
        """imagen → (texto_crudo, json_o_None). Lo que sale de aquí entra a su sistema como texto."""
        salida = preguntar(imagen, instruccion or INSTRUCCION_DOMINIO, system=VALVULA)
        return salida, parsear_json(salida)

### 2B · Patrón TOOL · CLIP clasifica (S11) y se registra como herramienta (S10)

In [ ]:
if PATRON == 'tool':
    from transformers import CLIPModel, CLIPProcessor
    CLIP_ID = 'openai/clip-vit-base-patch32'
    clip = CLIPModel.from_pretrained(CLIP_ID).to(device).eval(); cproc = CLIPProcessor.from_pretrained(CLIP_ID)

    @torch.no_grad()
    def emb_imagenes(lista):
        v = clip.get_image_features(**cproc(images=lista, return_tensors='pt').to(device)); return torch.nn.functional.normalize(v, dim=-1)
    @torch.no_grad()
    def emb_textos(lista):
        v = clip.get_text_features(**cproc(text=lista, return_tensors='pt', padding=True, truncation=True).to(device)); return torch.nn.functional.normalize(v, dim=-1)

    # 👉 TODO 2 · las clases de SU dominio como prompts (en inglés rinde mejor), y la descripción de la tool.
    CLASES = {'a photo of a cat': 'gato', 'a photo of a living room': 'sala', 'a person skiing': 'esqui'}   # ← cambien
    TOOL_SCHEMA = {'name': 'clasificar_imagen',
                   'description': 'Clasifica la foto adjunta en una de: ' + ', '.join(CLASES.values()) + '. Úsala SOLO si el usuario adjuntó una foto.',
                   'parameters': {'imagen_id': 'string'}}

    V_CLASES = emb_textos(list(CLASES))
    def ver(imagen):
        """imagen → {'clase': ..., 'coseno': ...}: lo que devuelve la tool al agente."""
        cos = (emb_imagenes([imagen]) @ V_CLASES.T)[0]
        k = int(cos.argmax())
        return {'clase': list(CLASES.values())[k], 'coseno': round(float(cos[k]), 3)}
    print('tool registrada:', TOOL_SCHEMA['name'])

### 2C · Patrón OUTPUT · Stable Diffusion dibuja (S12) · solo con GPU

In [ ]:
if PATRON == 'output':
    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
    pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5', torch_dtype=torch.float16).to('cuda')
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)   # la regla del paso (recap, duda 1)

    # 👉 TODO 2 · la plantilla de prompt de SU dominio (en inglés) y el negativo.
    def plantilla_prompt(texto_respuesta):
        return f'simple educational illustration of {texto_respuesta}, clean background, flat colors'   # ← cambien
    NEGATIVO = 'blurry, text, watermark, photo-realistic'

    def ver(prompt, seed=42, pasos=20):
        """prompt → imagen (PIL). Semilla fija: parte del experimento (S12)."""
        g = torch.Generator('cuda').manual_seed(seed)
        im = pipe(prompt, negative_prompt=NEGATIVO, num_inference_steps=pasos, guidance_scale=7.5, generator=g).images[0]
        return etiquetar(im)

    from PIL import ImageDraw
    def etiquetar(im):
        d = ImageDraw.Draw(im); d.rectangle([0, im.height - 22, im.width, im.height], fill=(0, 0, 0)); d.text((6, im.height - 18), 'IMAGEN GENERADA · SI4006', fill=(255, 255, 255)); return im

### 2D · Probar `ver()` sola, sobre 3 imágenes, con latencia

Antes de enchufar nada: ¿hace lo que prometieron? ¿Cuánto tarda?

In [ ]:
def cronometrar(f, *a, **k):
    t0 = time.time(); r = f(*a, **k); return r, round(time.time() - t0, 2)

if PATRON in ('input', 'tool'):
    for nombre, im in imgs.items():
        salida, seg = cronometrar(ver, im)
        print(f'[{nombre}] {seg}s →', salida if PATRON == 'tool' else salida[1] or salida[0][:200])
else:
    for texto in ['a right triangle with legs 3 and 4', 'a grey sofa in a small living room']:
        im, seg = cronometrar(ver, plantilla_prompt(texto))
        print(f'[{texto}] {seg}s'); display(im.resize((256, 256)))

# 👀 anoten: ¿la salida es lo prometido (texto / JSON parseable / clase / imagen)? ¿cuántos segundos? ¿dónde se equivocó?

## 3 · Parte 1 · El enchufe: `sistema_multimodal(pregunta, imagen)`

Aquí entra **su sistema de M3**. Lo ideal es importarlo desde su repo; si aún no lo tienen conectado, el notebook trae un **sustituto marcado** para que el pipeline corra de punta a punta (y lo dicen en el README).

In [ ]:
# 👉 TODO 3 · conecten su sistema de M3. Reemplacen esta función por la suya (misma firma):
#    responder_m3(pregunta: str, contexto_extra: str = '') -> str
def responder_m3(pregunta, contexto_extra=''):
    # SUSTITUTO (marcado): no hay RAG ni agente aquí. Solo para que el pipeline corra hoy.
    return f'[SUSTITUTO M3] Pregunta: {pregunta}. Contexto visual: {contexto_extra[:300]}'

def sistema_multimodal(pregunta, imagen=None):
    """pregunta (+ imagen) → respuesta. El enchufe depende del patrón."""
    if PATRON == 'input':                       # ANTES del RAG: lo visto se suma a la pregunta
        if imagen is None: return responder_m3(pregunta)
        crudo, js = ver(imagen)
        return responder_m3(pregunta, contexto_extra=json.dumps(js, ensure_ascii=False) if js else crudo)
    if PATRON == 'tool':                        # EN la lista de tools: el LLM decide llamarla
        # versión mínima del bucle de S10: si hay imagen Y la pregunta lo pide, llama la tool
        if imagen is not None and re.search(r'foto|imagen|esto|esta', pregunta, re.I):
            r = ver(imagen); return responder_m3(pregunta, contexto_extra=f"tool clasificar_imagen → {r}")
        return responder_m3(pregunta)
    if PATRON == 'output':                      # DESPUÉS del LLM: la respuesta incluye la imagen
        texto = responder_m3(pregunta)
        return texto, ver(plantilla_prompt(pregunta))

ejemplo = sistema_multimodal(PREGUNTA_USUARIO, list(imgs.values())[0] if PATRON != 'output' else None)
print(ejemplo if PATRON != 'output' else ejemplo[0]);
if PATRON == 'output': display(ejemplo[1].resize((256, 256)))

> 🔄 **Fin de la parte 1.** Cuatro cosas deben ser ciertas: `ver()` corre sola sobre 3 imágenes · la latencia está anotada · `sistema_multimodal` responde de punta a punta · ya vieron el primer fallo. Vuelvan al video.

---

## 4 · Parte 2 · Los casos con imagen

Mismas columnas del eval set de M2, una más: la imagen. Mínimo 5; al menos uno **adversarial** (lo esperado es que el sistema se niegue o diga "no se aprecia").

In [ ]:
# 👉 TODO 4 · ≥5 casos de SU dominio. criterio ∈ {'exacto', 'campos', 'valvula', 'juez_vlm', 'checklist', 'clip_score'}
#    esperado: str (exacto / valvula / juez_vlm) · dict (campos) · lista de ítems (checklist) · str prompt (clip_score)
CASOS = pd.DataFrame([
    {'id': 'img_01', 'pregunta': '¿Qué hay en esta foto?',              'imagen': 'gatos', 'esperado': 'gato',                                    'criterio': 'exacto',   'tipo': 'normal'},
    {'id': 'img_02', 'pregunta': 'Extrae los datos de la escena',       'imagen': 'sala',  'esperado': {'escena': 'interior'},                     'criterio': 'campos',   'tipo': 'normal'},
    {'id': 'img_03', 'pregunta': '¿Qué fecha aparece escrita abajo?',   'imagen': 'esqui', 'esperado': 'No se aprecia en la imagen.',              'criterio': 'valvula',  'tipo': 'adversarial'},
    {'id': 'img_04', 'pregunta': 'Describe lo que hace la persona',     'imagen': 'esqui', 'esperado': 'una persona esquiando en la nieve',       'criterio': 'juez_vlm', 'tipo': 'normal'},
    {'id': 'img_05', 'pregunta': 'Muéstrame un sofá gris en una sala',  'imagen': None,    'esperado': ['sofá', 'gris', 'sala'],                   'criterio': 'checklist','tipo': 'output'},
])
CASOS

## 5 · Parte 2 · El harness: criterios que saben mirar

La función de M2 no cambia: `calificar(caso, salida) → nota en [0, 1]`. Lo nuevo son tres criterios: **válvula**, **juez VLM** y **checklist**. Dos vienen hechos; uno lo completan ustedes.

In [ ]:
def nota_exacto(esperado, salida):
    return float(str(esperado).lower() in str(salida).lower())

def nota_campos(esperado: dict, salida):
    """fracción de campos esperados que aparecen con el valor correcto en la salida (texto o JSON)."""
    s = str(salida).lower()
    if isinstance(esperado, str): esperado = json.loads(esperado)
    ok = sum(1 for k, v in esperado.items() if str(v).lower() in s)
    return ok / max(1, len(esperado))

def nota_valvula(esperado, salida):
    return float(('no se aprecia' in str(salida).lower()) or ('no puedo' in str(salida).lower()))

def nota_juez_vlm(esperado, salida, imagen):
    """faithfulness visual: ¿lo que dijo el sistema se sostiene en la imagen? Si hay VLM, mira; si no, compara con la descripción gold."""
    if PATRON == 'input':
        veredicto = preguntar(imagen, f'Afirmación: "{str(salida)[:400]}". ¿Se sostiene en la imagen? Responde solo SI o NO.', max_new_tokens=3)
        return float(veredicto.strip().upper().startswith('S'))
    # sin VLM: juez de texto contra la descripción gold (muy burdo: solapamiento de palabras)
    g, s = set(str(esperado).lower().split()), set(str(salida).lower().split()); return len(g & s) / max(1, len(g))

def nota_checklist(items: list, salida_imagen):
    """para output: un VLM revisa ítem por ítem. Sin VLM cargado, pide revisión humana."""
    if 'preguntar' not in globals(): print('   (checklist: sin VLM cargado → anoten la revisión humana)'); return float('nan')
    ok = [preguntar(salida_imagen, f'¿Aparece "{it}" en la imagen? Responde solo SI o NO.', max_new_tokens=3).upper().startswith('S') for it in items]
    return sum(ok) / len(items)

def calificar(caso, salida, imagen=None):
    c, e = caso['criterio'], caso['esperado']
    if c == 'exacto':     return nota_exacto(e, salida)
    if c == 'campos':     return nota_campos(e, salida)
    if c == 'valvula':    return nota_valvula(e, salida)
    if c == 'juez_vlm':   return nota_juez_vlm(e, salida, imagen)
    if c == 'checklist':  return nota_checklist(e, salida[1] if isinstance(salida, tuple) else salida)
    if c == 'clip_score': return float('nan')   # ver S12: coseno imagen–prompt × 100, promedio de 3 semillas
    raise ValueError(c)

## 6 · Parte 2 · Correr el harness y llenar la tabla de delta

In [ ]:
filas = []
for _, caso in CASOS.iterrows():
    if PATRON == 'output' and caso['tipo'] != 'output': continue
    if PATRON != 'output' and caso['tipo'] == 'output': continue
    im = imgs.get(caso['imagen']) if caso['imagen'] else None
    salida, seg = cronometrar(sistema_multimodal, caso['pregunta'], im)
    nota = calificar(caso, salida, im)
    filas.append({'id': caso['id'], 'criterio': caso['criterio'], 'tipo': caso['tipo'], 'nota': nota, 'segundos': seg,
                  'salida': (salida if not isinstance(salida, tuple) else salida[0])[:120]})
RES = pd.DataFrame(filas); RES

In [ ]:
# 👉 TODO 6 · sus números de M3 (del scorecard que entregaron): promedio en los casos SIN imagen y latencia media.
M3 = {'casos_sin_imagen': 0.74, 'latencia_s': 2.1}   # ← cambien por los suyos

con = RES[RES.tipo != 'adversarial']; adv = RES[RES.tipo == 'adversarial']
delta = pd.DataFrame([
    {'grupo': 'casos sin imagen (M2)',     'M3': M3['casos_sin_imagen'], 'M4': '← corran su harness de M2 sobre sistema_multimodal(pregunta) sin imagen', 'lectura': '¿empeoró lo viejo?'},
    {'grupo': 'casos con imagen (nuevos)', 'M3': 0.0,                    'M4': round(con.nota.mean(), 2) if len(con) else None,                              'lectura': '¿se responden ahora? ¿cuál no?'},
    {'grupo': 'caso adversarial (válvula)','M3': '—',                    'M4': f"{int(adv.nota.sum())} / {len(adv)}" if len(adv) else '—',                 'lectura': '¿se negó cuando debía?'},
    {'grupo': 'latencia media por caso',   'M3': M3['latencia_s'],       'M4': round(RES.segundos.mean(), 2),                                                 'lectura': '¿cuánto cuesta ver?'},
    {'grupo': 'peor caso',                 'M3': '—',                    'M4': RES.sort_values('nota').iloc[0]['id'] if len(RES) else '—',                 'lectura': '¿dónde falla y por qué?'},
])
RES.to_csv('scorecard_m4.csv', index=False); print('scorecard_m4.csv guardado'); delta

> 👀 **Lean la tabla antes de celebrar.** Lo que se califica no es que M4 gane: es que la tabla exista, sea honesta y diga dónde falla. El peor caso es la fila más valiosa: ¿qué pasó? ¿qué harían?

### Opcional · SAM como tool: recortar antes de clasificar (profundización)

In [ ]:
# %pip install -q transformers  # SAM viene en transformers
# from transformers import SamModel, SamProcessor
# sam = SamModel.from_pretrained('facebook/sam-vit-base').to(device); sproc = SamProcessor.from_pretrained('facebook/sam-vit-base')
# im = list(imgs.values())[0]
# ent = sproc(im, input_points=[[[im.width // 2, im.height // 2]]], return_tensors='pt').to(device)   # un clic en el centro
# with torch.no_grad(): out = sam(**ent)
# masks = sproc.image_processor.post_process_masks(out.pred_masks.cpu(), ent['original_sizes'].cpu(), ent['reshaped_input_sizes'].cpu())[0][0, 0]
# recorte = Image.fromarray((masks.numpy()[..., None] * im).astype('uint8')); display(recorte.resize((256, 256)))
# # → pasen `recorte` a ver() y comparen la clase/coseno con la imagen completa: ¿sube la accuracy cuando el fondo confunde?

## Lo que se llevan (y lo que se entrega)

- **`ver()`** es una función con una responsabilidad; **el enchufe** depende del patrón (antes del RAG · en las tools · después del LLM); el sistema de M3 no se reescribe.
- **El harness no cambia**: criterios nuevos (válvula, juez VLM, checklist) en la misma función de M2. La **tabla de delta** responde tres preguntas: ¿mejoró lo nuevo? ¿no empeoró lo viejo? ¿cuánto cuesta?
- **Entrega M4** (dom 25 OCT · 23:59 · 10 %), carpeta `m4/`: este notebook terminado (o módulo) · `eval_set` con las imágenes (anonimizadas) · `scorecard_m4.csv` · `README.md` de una página con la pregunta, el patrón, el modelo, la tabla de delta, el peor caso y los límites.
- Las fotos de personas y documentos **no van al repo**. La imagen generada **se marca**. El VLM **describe, no diagnostica**.

*SI4006 · Universidad EAFIT · Sesión 13 · RESUELTO.*